In [1]:
# Uncomment only if your environment is missing any package
# %pip install -q openai python-dotenv pandas

import json
import re
import textwrap
from typing import Any, Dict, List, Optional
import os

import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI

# Load configuration from the project-level .env file.
# If this notebook is stored inside a notebooks/ folder, ../.env is typically correct.
load_dotenv("../.env", override=True)

AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
AZURE_OPENAI_MODEL = os.getenv("AZURE_OPENAI_MODEL")
AZURE_OPENAI_EMBEDDING_MODEL = os.getenv("AZURE_OPENAI_EMBEDDING_MODEL")
AZURE_OPENAI_MODEL_SECONDARY = os.getenv("AZURE_OPENAI_MODEL_SECONDARY")

if not AZURE_OPENAI_ENDPOINT or not AZURE_OPENAI_API_KEY or not AZURE_OPENAI_MODEL:
    raise ValueError(
        "Missing required Azure configuration. Check AZURE_OPENAI_ENDPOINT, "
        "AZURE_OPENAI_API_KEY, and AZURE_OPENAI_MODEL in ../.env"
    )

client = OpenAI(
    base_url=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY
)

print("Azure endpoint configured:", bool(AZURE_OPENAI_ENDPOINT))
print("API key configured       :", bool(AZURE_OPENAI_API_KEY))
print("Primary model deployment :", AZURE_OPENAI_MODEL)
print("Embedding deployment     :", AZURE_OPENAI_EMBEDDING_MODEL)
print("Secondary deployment     :", AZURE_OPENAI_MODEL_SECONDARY)

Azure endpoint configured: True
API key configured       : True
Primary model deployment : gpt-4.1
Embedding deployment     : text-embedding-3-small
Secondary deployment     : gpt-5-mini


In [2]:
from langchain_community.document_loaders import PyPDFLoader

files = [
    r"notebook\healthcare_policies\01_Gold_PPO_2026_Benefits_Authorization.pdf",
    r"notebook\healthcare_policies\02_Silver_HMO_2026_Benefits_Authorization.pdf",
    r"notebook\healthcare_policies\03_Advanced_Imaging_Utilization_Management_2026.pdf",
    r"notebook\healthcare_policies\04_Rehabilitation_Therapy_Policy_2026.pdf",
    r"notebook\healthcare_policies\05_Claims_Benefits_Appeals_Operations_2026.pdf"
]

documents = []

for file in files:
    loader = PyPDFLoader(file)
    documents.extend(loader.load())

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = splitter.split_documents(documents)

print(f"Total chunks: {len(chunks)}")

Total chunks: 11


In [10]:
from langchain_openai import AzureOpenAIEmbeddings

embeddings = AzureOpenAIEmbeddings(
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
    model="text-embedding-3-small",
    api_version="2024-02-01"
)

In [12]:
from langchain_chroma import Chroma

vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_db"
)



In [13]:
vector_store = Chroma(
    persist_directory="./chroma_db",
    embedding_function=embeddings
)

retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k":5}
)

docs = retriever.invoke(
    "What is the prior authorization process?"
)

for doc in docs:
    print(doc.page_content[:300])

imaging authorization.
SECTION 3: PHYSICAL THERAPY
The first 10 physical therapy visits in a benefit year do not require prior authorization. Prior authorization is
required starting with the 11th visit. The authorization request must include the diagnosis, functional goals,
progress to date and pro
imaging authorization.
SECTION 3: PHYSICAL THERAPY
The first 10 physical therapy visits in a benefit year do not require prior authorization. Prior authorization is
required starting with the 11th visit. The authorization request must include the diagnosis, functional goals,
progress to date and pro
imaging authorization.
SECTION 3: PHYSICAL THERAPY
The first 10 physical therapy visits in a benefit year do not require prior authorization. Prior authorization is
required starting with the 11th visit. The authorization request must include the diagnosis, functional goals,
progress to date and pro
SECTION 6: MEMBER RESPONSIBILITY
Prior authorization confirms that medical-necessity review requi

In [14]:
from langchain_openai import AzureChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

llm = AzureChatOpenAI(
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
    model=AZURE_OPENAI_MODEL,
    api_version="2024-02-01"
)

question = "What is the prior authorization process?"

retrieved_docs = retriever.invoke(question)

context = "\n\n".join(
    doc.page_content
    for doc in retrieved_docs
)

prompt = f"""
Answer the question using only the provided context.

Context:
{context}

Question:
{question}
"""

response = llm.invoke(prompt)

print(response.content)

The prior authorization process involves confirming that medical-necessity review requirements have been met before certain services are provided. For example, in physical therapy, prior authorization is required starting with the 11th visit in a benefit year, and the authorization request must include the diagnosis, functional goals, progress to date, and proposed treatment frequency. Prior authorization does not guarantee payment; final payment depends on eligibility, benefit limits, coding accuracy, and other plan provisions.


In [15]:
import sys
print(sys.version)

3.12.10 (tags/v3.12.10:0cc8128, Apr  8 2025, 12:21:36) [MSC v.1943 64 bit (AMD64)]
